In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
device

device(type='cuda')

# 2D UNet


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init

def init_weights(net, init_type='normal', gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and (classname.find('Conv') != -1 or classname.find('Linear') != -1):
            if init_type == 'normal':
                init.normal_(m.weight.data, 0.0, gain)
            elif init_type == 'xavier':
                init.xavier_normal_(m.weight.data, gain=gain)
            elif init_type == 'kaiming':
                init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
            elif init_type == 'orthogonal':
                init.orthogonal_(m.weight.data, gain=gain)
            else:
                raise NotImplementedError('initialization method [%s] is not implemented' % init_type)
            if hasattr(m, 'bias') and m.bias is not None:
                init.constant_(m.bias.data, 0.0)
        elif classname.find('BatchNorm2d') != -1:
            init.normal_(m.weight.data, 1.0, gain)
            init.constant_(m.bias.data, 0.0)

    print('initialize network with %s' % init_type)
    net.apply(init_func)

class conv_block(nn.Module):
    def __init__(self,ch_in,ch_out):
        super(conv_block,self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ch_in, ch_out, kernel_size=3,stride=1,padding=1,bias=True),
            nn.BatchNorm2d(ch_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_out, ch_out, kernel_size=3,stride=1,padding=1,bias=True),
            nn.BatchNorm2d(ch_out),
            nn.ReLU(inplace=True)
        )


    def forward(self,x):
        x = self.conv(x)
        return x

class up_conv(nn.Module):
    def __init__(self,ch_in,ch_out):
        super(up_conv,self).__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(ch_in,ch_out,kernel_size=3,stride=1,padding=1,bias=True),
		    nn.BatchNorm2d(ch_out),
			nn.ReLU(inplace=True)
        )

    def forward(self,x):
        x = self.up(x)
        return x

class Recurrent_block(nn.Module):
    def __init__(self,ch_out,t=2):
        super(Recurrent_block,self).__init__()
        self.t = t
        self.ch_out = ch_out
        self.conv = nn.Sequential(
            nn.Conv2d(ch_out,ch_out,kernel_size=3,stride=1,padding=1,bias=True),
		    nn.BatchNorm2d(ch_out),
			nn.ReLU(inplace=True)
        )

    def forward(self,x):
        for i in range(self.t):

            if i==0:
                x1 = self.conv(x)
            
            x1 = self.conv(x+x1)
        return x1
        
class RRCNN_block(nn.Module):
    def __init__(self,ch_in,ch_out,t=2):
        super(RRCNN_block,self).__init__()
        self.RCNN = nn.Sequential(
            Recurrent_block(ch_out,t=t),
            Recurrent_block(ch_out,t=t)
        )
        self.Conv_1x1 = nn.Conv2d(ch_in,ch_out,kernel_size=1,stride=1,padding=0)

    def forward(self,x):
        x = self.Conv_1x1(x)
        x1 = self.RCNN(x)
        return x+x1


class single_conv(nn.Module):
    def __init__(self,ch_in,ch_out):
        super(single_conv,self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ch_in, ch_out, kernel_size=3,stride=1,padding=1,bias=True),
            nn.BatchNorm2d(ch_out),
            nn.ReLU(inplace=True)
        )

    def forward(self,x):
        x = self.conv(x)
        return x

class Attention_block(nn.Module):
    def __init__(self,F_g,F_l,F_int):
        super(Attention_block,self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1,stride=1,padding=0,bias=True),
            nn.BatchNorm2d(F_int)
            )
        
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1,stride=1,padding=0,bias=True),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1,stride=1,padding=0,bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self,g,x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1+x1)
        psi = self.psi(psi)

        return x*psi


class U_Net(nn.Module):
    def __init__(self,img_ch=1,output_ch=1):
        super(U_Net,self).__init__()
        
        self.Maxpool = nn.MaxPool2d(kernel_size=2,stride=2)

        self.Conv1 = conv_block(ch_in=img_ch,ch_out=64)
        self.Conv2 = conv_block(ch_in=64,ch_out=128)
        self.Conv3 = conv_block(ch_in=128,ch_out=256)
        self.Conv4 = conv_block(ch_in=256,ch_out=512)
        self.Conv5 = conv_block(ch_in=512,ch_out=1024)

        self.Up5 = up_conv(ch_in=1024,ch_out=512)
        self.Up_conv5 = conv_block(ch_in=1024, ch_out=512)

        self.Up4 = up_conv(ch_in=512,ch_out=256)
        self.Up_conv4 = conv_block(ch_in=512, ch_out=256)
        
        self.Up3 = up_conv(ch_in=256,ch_out=128)
        self.Up_conv3 = conv_block(ch_in=256, ch_out=128)
        
        self.Up2 = up_conv(ch_in=128,ch_out=64)
        self.Up_conv2 = conv_block(ch_in=128, ch_out=64)

        self.Conv_1x1 = nn.Conv2d(64,output_ch,kernel_size=1,stride=1,padding=0)


    def forward(self,x):
        # encoding path
        x1 = self.Conv1(x)

        x2 = self.Maxpool(x1)
        x2 = self.Conv2(x2)
        
        x3 = self.Maxpool(x2)
        x3 = self.Conv3(x3)

        x4 = self.Maxpool(x3)
        x4 = self.Conv4(x4)

        x5 = self.Maxpool(x4)
        x5 = self.Conv5(x5)

        # decoding + concat path
        d5 = self.Up5(x5)
        d5 = torch.cat((x4,d5),dim=1)
        
        d5 = self.Up_conv5(d5)
        
        d4 = self.Up4(d5)
        d4 = torch.cat((x3,d4),dim=1)
        d4 = self.Up_conv4(d4)

        d3 = self.Up3(d4)
        d3 = torch.cat((x2,d3),dim=1)
        d3 = self.Up_conv3(d3)

        d2 = self.Up2(d3)
        d2 = torch.cat((x1,d2),dim=1)
        d2 = self.Up_conv2(d2)

        d1 = self.Conv_1x1(d2)

        return d1


class R2U_Net(nn.Module):
    def __init__(self,img_ch=1,output_ch=1,t=2):
        super(R2U_Net,self).__init__()
        
        self.Maxpool = nn.MaxPool2d(kernel_size=2,stride=2)
        self.Upsample = nn.Upsample(scale_factor=2)

        self.RRCNN1 = RRCNN_block(ch_in=img_ch,ch_out=64,t=t)

        self.RRCNN2 = RRCNN_block(ch_in=64,ch_out=128,t=t)
        
        self.RRCNN3 = RRCNN_block(ch_in=128,ch_out=256,t=t)
        
        self.RRCNN4 = RRCNN_block(ch_in=256,ch_out=512,t=t)
        
        self.RRCNN5 = RRCNN_block(ch_in=512,ch_out=1024,t=t)
        

        self.Up5 = up_conv(ch_in=1024,ch_out=512)
        self.Up_RRCNN5 = RRCNN_block(ch_in=1024, ch_out=512,t=t)
        
        self.Up4 = up_conv(ch_in=512,ch_out=256)
        self.Up_RRCNN4 = RRCNN_block(ch_in=512, ch_out=256,t=t)
        
        self.Up3 = up_conv(ch_in=256,ch_out=128)
        self.Up_RRCNN3 = RRCNN_block(ch_in=256, ch_out=128,t=t)
        
        self.Up2 = up_conv(ch_in=128,ch_out=64)
        self.Up_RRCNN2 = RRCNN_block(ch_in=128, ch_out=64,t=t)

        self.Conv_1x1 = nn.Conv2d(64,output_ch,kernel_size=1,stride=1,padding=0)


    def forward(self,x):
        # encoding path
        x1 = self.RRCNN1(x)

        x2 = self.Maxpool(x1)
        x2 = self.RRCNN2(x2)
        
        x3 = self.Maxpool(x2)
        x3 = self.RRCNN3(x3)

        x4 = self.Maxpool(x3)
        x4 = self.RRCNN4(x4)

        x5 = self.Maxpool(x4)
        x5 = self.RRCNN5(x5)

        # decoding + concat path
        d5 = self.Up5(x5)
        d5 = torch.cat((x4,d5),dim=1)
        d5 = self.Up_RRCNN5(d5)
        
        d4 = self.Up4(d5)
        d4 = torch.cat((x3,d4),dim=1)
        d4 = self.Up_RRCNN4(d4)

        d3 = self.Up3(d4)
        d3 = torch.cat((x2,d3),dim=1)
        d3 = self.Up_RRCNN3(d3)

        d2 = self.Up2(d3)
        d2 = torch.cat((x1,d2),dim=1)
        d2 = self.Up_RRCNN2(d2)

        d1 = self.Conv_1x1(d2)

        return d1



class AttU_Net(nn.Module):
    def __init__(self,img_ch=1,output_ch=1):
        super(AttU_Net,self).__init__()
        
        self.Maxpool = nn.MaxPool2d(kernel_size=2,stride=2)

        self.Conv1 = conv_block(ch_in=img_ch,ch_out=64)
        self.Conv2 = conv_block(ch_in=64,ch_out=128)
        self.Conv3 = conv_block(ch_in=128,ch_out=256)
        self.Conv4 = conv_block(ch_in=256,ch_out=512)
        self.Conv5 = conv_block(ch_in=512,ch_out=1024)

        self.Up5 = up_conv(ch_in=1024,ch_out=512)
        self.Att5 = Attention_block(F_g=512,F_l=512,F_int=256)
        self.Up_conv5 = conv_block(ch_in=1024, ch_out=512)

        self.Up4 = up_conv(ch_in=512,ch_out=256)
        self.Att4 = Attention_block(F_g=256,F_l=256,F_int=128)
        self.Up_conv4 = conv_block(ch_in=512, ch_out=256)
        
        self.Up3 = up_conv(ch_in=256,ch_out=128)
        self.Att3 = Attention_block(F_g=128,F_l=128,F_int=64)
        self.Up_conv3 = conv_block(ch_in=256, ch_out=128)
        
        self.Up2 = up_conv(ch_in=128,ch_out=64)
        self.Att2 = Attention_block(F_g=64,F_l=64,F_int=32)
        self.Up_conv2 = conv_block(ch_in=128, ch_out=64)

        self.Conv_1x1 = nn.Conv2d(64,output_ch,kernel_size=1,stride=1,padding=0)


    def forward(self,x):
        # encoding path
        x1 = self.Conv1(x)

        x2 = self.Maxpool(x1)
        x2 = self.Conv2(x2)
        
        x3 = self.Maxpool(x2)
        x3 = self.Conv3(x3)

        x4 = self.Maxpool(x3)
        x4 = self.Conv4(x4)

        x5 = self.Maxpool(x4)
        x5 = self.Conv5(x5)

        # decoding + concat path
        d5 = self.Up5(x5)
        x4 = self.Att5(g=d5,x=x4)
        d5 = torch.cat((x4,d5),dim=1)        
        d5 = self.Up_conv5(d5)
        
        d4 = self.Up4(d5)
        x3 = self.Att4(g=d4,x=x3)
        d4 = torch.cat((x3,d4),dim=1)
        d4 = self.Up_conv4(d4)

        d3 = self.Up3(d4)
        x2 = self.Att3(g=d3,x=x2)
        d3 = torch.cat((x2,d3),dim=1)
        d3 = self.Up_conv3(d3)

        d2 = self.Up2(d3)
        x1 = self.Att2(g=d2,x=x1)
        d2 = torch.cat((x1,d2),dim=1)
        d2 = self.Up_conv2(d2)

        d1 = self.Conv_1x1(d2)

        return d1


class R2AttU_Net(nn.Module):
    def __init__(self,img_ch=1,output_ch=1,t=2):
        super(R2AttU_Net,self).__init__()
        
        self.Maxpool = nn.MaxPool2d(kernel_size=2,stride=2)
        self.Upsample = nn.Upsample(scale_factor=2)

        self.RRCNN1 = RRCNN_block(ch_in=img_ch,ch_out=64,t=t)

        self.RRCNN2 = RRCNN_block(ch_in=64,ch_out=128,t=t)
        
        self.RRCNN3 = RRCNN_block(ch_in=128,ch_out=256,t=t)
        
        self.RRCNN4 = RRCNN_block(ch_in=256,ch_out=512,t=t)
        
        self.RRCNN5 = RRCNN_block(ch_in=512,ch_out=1024,t=t)
        

        self.Up5 = up_conv(ch_in=1024,ch_out=512)
        self.Att5 = Attention_block(F_g=512,F_l=512,F_int=256)
        self.Up_RRCNN5 = RRCNN_block(ch_in=1024, ch_out=512,t=t)
        
        self.Up4 = up_conv(ch_in=512,ch_out=256)
        self.Att4 = Attention_block(F_g=256,F_l=256,F_int=128)
        self.Up_RRCNN4 = RRCNN_block(ch_in=512, ch_out=256,t=t)
        
        self.Up3 = up_conv(ch_in=256,ch_out=128)
        self.Att3 = Attention_block(F_g=128,F_l=128,F_int=64)
        self.Up_RRCNN3 = RRCNN_block(ch_in=256, ch_out=128,t=t)
        
        self.Up2 = up_conv(ch_in=128,ch_out=64)
        self.Att2 = Attention_block(F_g=64,F_l=64,F_int=32)
        self.Up_RRCNN2 = RRCNN_block(ch_in=128, ch_out=64,t=t)

        self.Conv_1x1 = nn.Conv2d(64,output_ch,kernel_size=1,stride=1,padding=0)


    def forward(self,x):
        # encoding path
        x1 = self.RRCNN1(x)

        x2 = self.Maxpool(x1)
        x2 = self.RRCNN2(x2)
        
        x3 = self.Maxpool(x2)
        x3 = self.RRCNN3(x3)

        x4 = self.Maxpool(x3)
        x4 = self.RRCNN4(x4)

        x5 = self.Maxpool(x4)
        x5 = self.RRCNN5(x5)

        # decoding + concat path
        d5 = self.Up5(x5)
        x4 = self.Att5(g=d5,x=x4)
        d5 = torch.cat((x4,d5),dim=1)
        d5 = self.Up_RRCNN5(d5)
        
        d4 = self.Up4(d5)
        x3 = self.Att4(g=d4,x=x3)
        d4 = torch.cat((x3,d4),dim=1)
        d4 = self.Up_RRCNN4(d4)

        d3 = self.Up3(d4)
        x2 = self.Att3(g=d3,x=x2)
        d3 = torch.cat((x2,d3),dim=1)
        d3 = self.Up_RRCNN3(d3)

        d2 = self.Up2(d3)
        x1 = self.Att2(g=d2,x=x1)
        d2 = torch.cat((x1,d2),dim=1)
        d2 = self.Up_RRCNN2(d2)

        d1 = self.Conv_1x1(d2)

        return d1

In [4]:
import os
import torch
import nibabel as nib
import numpy as np
import cv2
from torch.utils.data import Dataset

def build_image_mask_pairs(image_dir, mask_dir):
    image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".nii")])
    mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith(".nii")])

    # Normalize image names to match with masks
    def normalize(name):
        if name.startswith("coronacases_org_"):
            return name.replace("coronacases_org_", "coronacases_").replace(".nii", "")
        if name.startswith("radiopaedia_org_covid-19-pneumonia-"):
            name = name.replace("radiopaedia_org_covid-19-pneumonia-", "")
            name = name.replace("-dcm", "")
            return "radiopaedia_" + name.replace(".nii", "")
        return name.replace(".nii", "")

    image_mask_pairs = []
    for img in image_files:
        norm_name = normalize(img)
        mask_filename = norm_name + ".nii"
        if mask_filename in mask_files:
            image_mask_pairs.append((img, mask_filename))
    return image_mask_pairs

class COVID2DDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_mask_pairs, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_mask_pairs = image_mask_pairs
        self.transform = transform
        self.target_size = (256, 256)
        self.slices = []

        # Pre-index all slice references (image, mask, slice index)
        for img_file, mask_file in self.image_mask_pairs:
            img_path = os.path.join(image_dir, img_file)
            img_nii = nib.load(img_path)
            num_slices = img_nii.shape[2]
            for idx in range(num_slices):
                self.slices.append((img_file, mask_file, idx))

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        img_file, mask_file, slice_idx = self.slices[idx]
        img = nib.load(os.path.join(self.image_dir, img_file)).get_fdata()
        mask = nib.load(os.path.join(self.mask_dir, mask_file)).get_fdata()

        img_slice = img[:, :, slice_idx]
        mask_slice = mask[:, :, slice_idx]

        # Resize and normalize
        img_resized = cv2.resize(img_slice, self.target_size, interpolation=cv2.INTER_AREA)
        img_resized = (img_resized - np.min(img_resized)) / (np.max(img_resized) - np.min(img_resized) + 1e-8)

        mask_resized = cv2.resize(mask_slice, self.target_size, interpolation=cv2.INTER_NEAREST)
        mask_resized = (mask_resized > 0).astype(np.uint8)

        # Add channel dim
        img_tensor = torch.FloatTensor(img_resized).unsqueeze(0)  # [1, H, W]
        mask_tensor = torch.LongTensor(mask_resized).unsqueeze(0)  # [1, H, W]

        return img_tensor, mask_tensor

image_dir = "/kaggle/input/covid19-ct-scans/ct_scans"
mask_dir = "/kaggle/input/covid19-ct-scans/infection_mask"

pairs = build_image_mask_pairs(image_dir, mask_dir)

# Split into train/val
from sklearn.model_selection import train_test_split
train_pairs, val_test = train_test_split(pairs, test_size=0.2, random_state=42)
val_pairs, test_pairs = train_test_split(val_test, test_size=0.5, random_state=42)

train_ds = COVID2DDataset(image_dir, mask_dir, train_pairs)
val_ds   = COVID2DDataset(image_dir, mask_dir, val_pairs)
test_ds = COVID2DDataset(image_dir, mask_dir, test_pairs)

from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4)

In [5]:
print(f'Train Count: {len(train_loader)*8} \nTest Count: {len(test_loader)} \nVal Count: {len(val_loader)*8}')

Train Count: 2888 
Test Count: 293 
Val Count: 344


# 2D Dataloader Function

In [6]:
# import os
# import torch
# import numpy as np
# import nibabel as nib
# import cv2
# from torch.utils.data import Dataset

# class LungDecathlon2DDataset(Dataset):
#     def __init__(self, image_dir, mask_dir, file_list, transform=None):
#         self.image_dir = image_dir
#         self.mask_dir = mask_dir
#         self.file_list = file_list          # list of volume filenames
#         self.transform = transform
#         self.target_size = (256, 256)

#         # Preload and slice volumes to speed up access
#         self.slices = []  # each item: (nii_filename, slice_index)
#         for fname in self.file_list:
#             img = nib.load(os.path.join(self.image_dir, fname))
#             num_slices = img.shape[2]
#             for idx in range(num_slices):
#                 self.slices.append((fname, idx))

#     def __len__(self):
#         return len(self.slices)

#     def __getitem__(self, idx):
#         fname, slice_idx = self.slices[idx]
#         img_path = os.path.join(self.image_dir, fname)
#         mask_path = os.path.join(self.mask_dir, fname)

#         # Load 3D volumes
#         img_nii = nib.load(img_path); img_vol = img_nii.get_fdata()
#         mask_nii = nib.load(mask_path); mask_vol = mask_nii.get_fdata()

#         # Extract the specific 2D slice
#         img_slice = img_vol[:, :, slice_idx]
#         mask_slice = mask_vol[:, :, slice_idx]

#         # Resize to target_size
#         img2d = cv2.resize(img_slice, self.target_size, interpolation=cv2.INTER_AREA)
#         mask2d = cv2.resize(mask_slice, self.target_size, interpolation=cv2.INTER_NEAREST)

#         # Normalize image to [0,1]
#         img2d = (img2d - img2d.min()) / (img2d.max() - img2d.min() + 1e-8)
#         img2d = img2d.astype(np.float32)

#         # Binarize mask
#         mask2d = (mask2d > 0).astype(np.uint8)

#         # Add channel dimension
#         img2d = np.expand_dims(img2d, 0)  # shape [1, 256, 256]
#         mask2d = np.expand_dims(mask2d, 0)  # shape [1, 256, 256]

#         # Apply any augmentations/transforms
#         if self.transform:
#             sample = {"image": img2d.transpose(1,2,0), "mask": mask2d.transpose(1,2,0)}
#             aug = self.transform(image=sample["image"], mask=sample["mask"])
#             img2d = aug["image"].transpose(2,0,1)
#             mask2d = aug["mask"].transpose(2,0,1)

#         return torch.FloatTensor(img2d), torch.LongTensor(mask2d)
# from sklearn.model_selection import train_test_split
# img_path = '/kaggle/input/medical-segmentation-decathlon-lung/imagesTr'
# lbl_path = '/kaggle/input/medical-segmentation-decathlon-lung/labelsTr'

# img_path = '/kaggle/input/covid19-ct-scans/ct_scans'
# lbl_path = '/kaggle/input/covid19-ct-scans/infection_mask'

# # Collect training volume files
# all_files = sorted(os.listdir(img_path))
# train_vols, val_vols = train_test_split(all_files, test_size=0.2, random_state=42)

# # Create datasets
# train_ds = LungDecathlon2DDataset(img_path, lbl_path, train_vols, transform=None)
# val_ds   = LungDecathlon2DDataset(img_path, lbl_path, val_vols, transform=None)

# # DataLoaders
# from torch.utils.data import DataLoader
# train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
# val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4)


In [7]:
# import os
# from torch.utils.data import Dataset
# from PIL import Image
# import numpy as np
# import torch
# from sklearn.model_selection import StratifiedShuffleSplit


# class SegmentationDataset(Dataset):
#     def __init__(self, images_dir, masks_dir, image_size=(256, 256), transform=None):
#         self.images_dir = images_dir
#         self.masks_dir = masks_dir
#         self.image_size = image_size
#         self.transform = transform

#         self.image_files = sorted(os.listdir(images_dir))
#         self.mask_files = sorted(os.listdir(masks_dir))

#         # Extract class labels from filenames
#         self.labels = []
#         for img_name in self.image_files:
#             if img_name.startswith('lungsMD'):
#                 self.labels.append(1)  # Lung Cancer
#             elif img_name.startswith('lungs'):
#                 self.labels.append(2)  # Covid-19
#             else:
#                 self.labels.append(0)  # Normal

#     def __len__(self):
#         return len(self.image_files)

#     def __getitem__(self, idx):
#         img_name = self.image_files[idx]
#         mask_name = self.mask_files[idx]

#         # assert img_name.split('.')[0] == mask_name.split('_mask')[0], \
#         #     f"Mismatched image and mask: {img_name}, {mask_name}"

#         img_path = os.path.join(self.images_dir, img_name)
#         mask_path = os.path.join(self.masks_dir, mask_name)

#         # Load image and mask
#         image = Image.open(img_path).convert("L")
#         mask = Image.open(mask_path).convert("L")

#         # Resize
#         image = image.resize(self.image_size, Image.BILINEAR)
#         mask = mask.resize(self.image_size, Image.NEAREST)

#         # Convert to arrays
#         image = np.array(image, dtype=np.float32)
#         image = (image - image.min()) / (image.max() - image.min() + 1e-8)
#         mask = np.array(mask, dtype=np.int64)/255.0

#         # Relabel mask based on class
#         if 'lungMD' in img_name:
#             pass  # keep 1
#         elif img_name.startswith('lung'):
#             mask[mask == 1] = 2  # relabel 1 -> 2 for covid
#         else:
#             pass  # normal stays zero

#         # To tensor
#         image_tensor = torch.from_numpy(image).unsqueeze(0)
#         mask_tensor = torch.from_numpy(mask)

#         # Apply transforms
#         if self.transform:
#             image_tensor = self.transform(image_tensor)
#             mask_tensor = self.transform(mask_tensor)

#         return image_tensor, mask_tensor

#     def get_labels(self):
#         return self.labels

In [8]:
# from torch.utils.data import Subset, DataLoader
# from sklearn.model_selection import train_test_split
# from collections import Counter

# def get_dataloaders(dataset, batch_size=4, stratify=True, val_size=0.15, test_size=0.2):
#     """
#     Returns stratified train, validation, and test dataloaders.
#     """
#     if not stratify:
#         # Simple random split without stratification
#         dataset_size = len(dataset)
#         train_len = int(dataset_size * (1 - val_size - test_size))
#         val_len = int(dataset_size * val_size)
#         test_len = dataset_size - train_len - val_len
#         train_subset, val_subset, test_subset = torch.utils.data.random_split(
#             dataset, [train_len, val_len, test_len]
#         )
#     else:
#         # Get labels for stratification
#         y = dataset.get_labels()

#         # First split: train + val and test
#         train_val_indices, test_indices = train_test_split(
#             np.arange(len(y)),
#             test_size=test_size,
#             stratify=y,
#             random_state=42
#         )

#         # Second split: train and val
#         y_train_val = np.array(y)[train_val_indices]
#         train_indices, val_indices = train_test_split(
#             train_val_indices,
#             test_size=val_size / (1 - test_size),
#             stratify=y_train_val,
#             random_state=42
#         )

#         # Create subsets
#         train_subset = Subset(dataset, train_indices)
#         val_subset = Subset(dataset, val_indices)
#         test_subset = Subset(dataset, test_indices)

#     # Create DataLoaders
#     train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
#     test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

#     print(f"Train size: {len(train_subset)}, Val size: {len(val_subset)}, Test size: {len(test_subset)}")
#     return train_loader, val_loader, test_loader

In [9]:

# dataset = SegmentationDataset(
#     images_dir='/kaggle/input/iit-kgp-lung-task-part-1-data/dataset/images',
#     masks_dir='/kaggle/input/iit-kgp-lung-task-part-1-data/dataset/masks',
#     image_size=(256, 256)
# )


# train_loader, val_loader, test_loader = get_dataloaders(dataset, batch_size=16, stratify=True, val_size=0.15, test_size=0.2)

In [10]:
# from collections import Counter

# def show_class_distribution(loader, dataset):
#     indices = loader.dataset.indices
#     labels = [dataset.labels[i] for i in indices]
#     print("Class distribution:", Counter(labels))

# show_class_distribution(train_loader, dataset)
# show_class_distribution(test_loader, dataset)
# show_class_distribution(val_loader, dataset)

# Metrics Calculations

In [11]:
# import torch

# def dice_score(preds, targets, num_classes=3, smooth=1e-6):
#     """
#     preds: [B, H, W], predicted class indices
#     targets: [B, H, W], ground truth class indices
#     """
#     dice_scores = []

#     for cls in range(num_classes):
#         pred_mask = (preds == cls)
#         target_mask = (targets == cls)

#         intersection = (pred_mask & target_mask).sum()
#         total = (pred_mask.sum() + target_mask.sum())

#         dice = (2. * intersection + smooth) / (total + smooth)
#         dice_scores.append(dice.item())

#     return dice_scores  # List of dice per class


# def iou_score(preds, targets, num_classes=3, smooth=1e-6):
#     """
#     preds: [B, H, W], predicted class indices
#     targets: [B, H, W], ground truth class indices
#     """
#     iou_scores = []

#     for cls in range(num_classes):
#         pred_mask = (preds == cls)
#         target_mask = (targets == cls)

#         intersection = (pred_mask & target_mask).sum()
#         union = (pred_mask.sum() + target_mask.sum() - intersection)

#         iou = (intersection + smooth) / (union + smooth)
#         iou_scores.append(iou.item())

#     return iou_scores  # List of IoU per class

# Model Training

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

# --- Binary metrics ---
def binary_iou(pred, target):
    pred, target = pred.view(-1), target.view(-1)
    intersection = (pred & target).float().sum().item()
    union = pred.float().sum().item() + target.float().sum().item() - intersection
    return intersection / union if union != 0 else np.nan

def binary_dice(pred, target):
    pred, target = pred.view(-1), target.view(-1)
    intersection = (pred & target).float().sum().item()
    return (2. * intersection) / (pred.float().sum().item() + target.float().sum().item() + 1e-8)

def binary_f1(pred, target):
    pred, target = pred.view(-1), target.view(-1)
    tp = (pred & target).sum().item()
    fp = (pred & (~target)).sum().item()
    fn = ((~pred) & target).sum().item()
    return (2 * tp) / (2 * tp + fp + fn + 1e-8) if (tp + fp + fn) != 0 else np.nan

# --- Training function ---
def train_binary_segmentation(model, train_loader, val_loader, num_epochs, device, save_path):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    model.to(device)

    best_val_dice = 0.0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        iou_scores, dice_scores, f1_scores = [], [], []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device).float()

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).bool()

            iou_scores.append(binary_iou(preds, masks.bool()))
            dice_scores.append(binary_dice(preds, masks.bool()))
            f1_scores.append(binary_f1(preds, masks.bool()))

            pbar.set_postfix({
                'Loss': f"{train_loss / (len(iou_scores)):.4f}",
                'Dice': f"{np.nanmean(dice_scores):.4f}",
                'IoU': f"{np.nanmean(iou_scores):.4f}",
                'F1': f"{np.nanmean(f1_scores):.4f}"
            }, refresh=True)

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        val_ious, val_dices, val_f1s = [], [], []

        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
                images = images.to(device)
                masks = masks.to(device).float()

                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item()

                preds = (torch.sigmoid(outputs) > 0.5).bool()

                val_ious.append(binary_iou(preds, masks.bool()))
                val_dices.append(binary_dice(preds, masks.bool()))
                val_f1s.append(binary_f1(preds, masks.bool()))

        avg_val_dice = np.nanmean(val_dices)

        print(f"\nEpoch {epoch+1} Summary:")
        print(f"Train Loss: {train_loss / len(train_loader):.4f} | Val Loss: {val_loss / len(val_loader):.4f}")
        print(f"Val IoU: {np.nanmean(val_ious):.4f} | Dice: {avg_val_dice:.4f} | F1: {np.nanmean(val_f1s):.4f}")

        # Save best model
        if avg_val_dice > best_val_dice:
            best_val_dice = avg_val_dice
            torch.save(model.state_dict(), f"{save_path}/best_model.pth")
            print("✅ Best model saved!")

    torch.save(model.state_dict(), f"{save_path}/final_model.pth")
    print("✅ Final model saved.")


model = U_Net()

train_binary_segmentation(model, train_loader, val_loader, num_epochs=30, device=torch.device('cuda'), save_path='')

Epoch 1/30 [Val]: 100%|██████████| 43/43 [00:50<00:00,  1.18s/it]



Epoch 1 Summary:
Train Loss: 0.2577 | Val Loss: 0.1718
Val IoU: 0.0723 | Dice: 0.0908 | F1: 0.1148
✅ Best model saved!


Epoch 2/30 [Val]: 100%|██████████| 43/43 [00:48<00:00,  1.14s/it]



Epoch 2 Summary:
Train Loss: 0.1317 | Val Loss: 0.1024
Val IoU: 0.1305 | Dice: 0.1489 | F1: 0.1940
✅ Best model saved!


Epoch 3/30 [Val]: 100%|██████████| 43/43 [00:48<00:00,  1.13s/it]


...


Epoch 28 Summary:
Train Loss: 0.0031 | Val Loss: 0.0247
Val IoU: 0.2917 | Dice: 0.2568 | F1: 0.4089


Epoch 29/30 [Val]: 100%|██████████| 43/43 [00:49<00:00,  1.15s/it]



Epoch 29 Summary:
Train Loss: 0.0031 | Val Loss: 0.0254
Val IoU: 0.3003 | Dice: 0.2737 | F1: 0.4203


Epoch 30/30 [Val]: 100%|██████████| 43/43 [00:48<00:00,  1.13s/it]



Epoch 30 Summary:
Train Loss: 0.0029 | Val Loss: 0.0240
Val IoU: 0.3299 | Dice: 0.3052 | F1: 0.4525
✅ Final model saved.


In [13]:
# --- Testing Function ---
def test_binary_model(model, test_loader, device='cuda', save_preds=True, save_dir="preds"):
    model.eval()
    model.to(device)

    iou_scores, dice_scores, f1_scores = [], [], []

    if save_preds:
        os.makedirs(save_dir, exist_ok=True)

    with torch.no_grad():
        for idx, (images, masks) in enumerate(tqdm(test_loader, desc="Testing")):
            images = images.to(device)
            masks = masks.to(device).float()

            outputs = model(images)
            preds = (torch.sigmoid(outputs) > 0.5).bool()

            iou_scores.append(binary_iou(preds, masks.bool()))
            dice_scores.append(binary_dice(preds, masks.bool()))
            f1_scores.append(binary_f1(preds, masks.bool()))

            if save_preds:
                for b in range(images.size(0)):
                    pred_np = preds[b][0].cpu().numpy().astype(np.uint8) * 255
                    mask_np = masks[b][0].cpu().numpy().astype(np.uint8) * 255
                    img_np = images[b][0].cpu().numpy() * 255

                    cv2.imwrite(os.path.join(save_dir, f"img_{idx}_{b}.png"), img_np)
                    cv2.imwrite(os.path.join(save_dir, f"gt_{idx}_{b}.png"), mask_np)
                    cv2.imwrite(os.path.join(save_dir, f"pred_{idx}_{b}.png"), pred_np)

    print("\n✅ Test Results:")
    print(f"IoU:  {np.nanmean(iou_scores):.4f}")
    print(f"Dice: {np.nanmean(dice_scores):.4f}")
    print(f"F1:   {np.nanmean(f1_scores):.4f}")
    
test_binary_model(model, test_loader, device='cuda', save_preds=True, save_dir="preds")

Testing: 100%|██████████| 293/293 [00:32<00:00,  9.15it/s]


✅ Test Results:
IoU:  0.4918
Dice: 0.4745
F1:   0.6044


In [14]:
NikhilIsTheBest

NameError: name 'NikhilIsTheBest' is not defined

## 2D UNet Model Training

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceLoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        # flatten label and prediction tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        # intersection is equivalent to True Positive count
        # union is the addition of both prediction and target sizes
        intersection = (inputs * targets).sum()
        dice = (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)

        return 1 - dice

In [ ]:
from collections import Counter

def compute_class_weights(loader, dataset, device, num_classes=3, weighted_loss='inverse'):
    indices = loader.dataset.indices
    labels = [dataset.labels[i] for i in indices]
    label_counter = Counter(labels)
    class_counts = np.array([label_counter.get(i, 0) for i in range(num_classes)])
    print("Class counts:", dict(enumerate(class_counts)))
    if weighted_loss == 'inverse':
        class_weights = 1.0 / class_counts
    elif weighted_loss == 'log':
        class_weights = 1.0 / np.log(class_counts + 1.0)
    else:
        raise ValueError("Unsupported weighting method")
    class_weights = class_weights / class_weights.sum() * num_classes
    return torch.tensor(class_weights, dtype=torch.float32).to(device)

In [ ]:
# # import pandas as pd
# from tqdm import tqdm
# import torch.optim as optim
# import os
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# optimizer = optim.Adam(model.parameters(), lr=1e-4)
# dataloaders = {
#     'train': test_loader,
#     'val': val_loader
# }
# # criterion = DiceLoss()
# class_weights = compute_class_weights(dataloaders['train'], dataset, device, num_classes=3, weighted_loss='inverse')
# criterion = nn.CrossEntropyLoss()

# num_epochs=2
# results = {
#     'epoch': [],
#     'train_loss': [], 'val_loss': [],
#     'train_dice': [], 'val_dice': [],
#     'train_iou': [], 'val_iou': []
# }

# model = model.to(device)

# for epoch in range(num_epochs):
#     print(f"Epoch {epoch+1}/{num_epochs}")
#     print("-" * 30)

#     for phase in ['train','val']:
#         if phase == 'train':
#             model.train()
#         else:
#             model.eval()

#         running_loss = []
#         running_dice=[]
#         running_iou=[]
#         all_probs = []
#         all_targets = []

#         for images, masks in tqdm(dataloaders[phase], desc=phase):
#             images = images.to(device)
#             masks = masks.to(device)

#             with torch.set_grad_enabled(phase == 'train'):
#                 outputs = model(images)
#                 loss = criterion(outputs, masks.long())

#                 if phase == 'train':
#                     optimizer.zero_grad()
#                     loss.backward()
#                     optimizer.step()

#             # Threshold predictions
#             probs = torch.sigmoid(outputs)
#             preds = (probs > 0.5).float()

#             # Save for AUC
#             all_probs.append(probs.cpu())
#             all_targets.append(masks.cpu())
#             # Compute metrics
#             running_loss.append(loss.item())
#             running_dice.append(dice_score(preds, masks).mean().item())
#             running_iou.append(iou_score(preds, masks).mean().item())


#         # End of batch loop

#         # Compute epoch-level metrics
#         epoch_loss = np.mean(running_loss)
#         epoch_dice =np.mean(running_dice)
#         epoch_iou = np.mean(running_iou)
    
#         results[f'{phase}_loss'].append(epoch_loss)
#         results[f'{phase}_dice'].append(epoch_dice)
#         results[f'{phase}_iou'].append(epoch_iou)
#         results['epoch'].append(epoch + 1)

#         print(f"{phase.upper()} Loss: {epoch_loss:.4f} | Dice: {epoch_dice:.4f} | IoU: {epoch_iou:.4f} ")

#     print()

# # End of epoch loop



In [ ]:
from tqdm import tqdm
import torch.optim as optim
import numpy as np

# Define Dice and IoU functions (same as before)
def dice_score(preds, targets, num_classes=3, smooth=1e-6):
    """Returns mean Dice score across all classes"""
    dice_scores = []
    preds = preds.view(-1)
    targets = targets.view(-1)
    
    for cls in range(num_classes):
        pred_mask = (preds == cls)
        target_mask = (targets == cls)
        intersection = (pred_mask & target_mask).sum()
        total = pred_mask.sum() + target_mask.sum()
        dice = (2. * intersection + smooth) / (total + smooth)
        dice_scores.append(dice.item())
    
    return np.mean(dice_scores)

def iou_score(preds, targets, num_classes=3, smooth=1e-6):
    """Returns mean IoU across all classes"""
    iou_scores = []
    preds = preds.view(-1)
    targets = targets.view(-1)
    
    for cls in range(num_classes):
        pred_mask = (preds == cls)
        target_mask = (targets == cls)
        intersection = (pred_mask & target_mask).sum()
        union = pred_mask.sum() + target_mask.sum() - intersection
        iou = (intersection + smooth) / (union + smooth)
        iou_scores.append(iou.item())
    
    return np.mean(iou_scores)
def log_gpu_stats(tag=""):
    dev = torch.cuda.current_device()
    cur = torch.cuda.memory_allocated(dev) / 1e9
    peak = torch.cuda.max_memory_allocated(dev) / 1e9
    info = nvmlDeviceGetMemoryInfo(handle)
    # print(f"[GPU][{tag}] PYTORCH alloc: {cur:.3f} GB | peak: {peak:.3f} GB | "
    #       f"TOTAL used: {info.used/1e9:.3f} GB / {info.total/1e9:.3f} GB")
    torch.cuda.reset_peak_memory_stats(dev)


dataloaders = {
    'train': test_loader,
    'val': val_loader
}
# criterion = DiceLoss()
class_weights = compute_class_weights(dataloaders['train'], dataset, device, num_classes=3, weighted_loss='inverse')

# Training loop with progress bar updates
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights).to(device)  # Add class weights if needed

num_epochs = 20
results = {
    'epoch': [],
    'train_loss': [], 'val_loss': [],
    'train_dice': [], 'val_dice': [],
    'train_iou': [], 'val_iou': []
}

model = model.to(device)


from pynvml import nvmlInit, nvmlDeviceGetHandleByIndex, nvmlDeviceGetMemoryInfo
nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)  # for GPU 0
mem_info = nvmlDeviceGetMemoryInfo(handle)


for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 30)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = []
        running_dice = []
        running_iou = []

        # Create progress bar
        pbar = tqdm(dataloaders[phase], desc=f"{phase.upper()} Phase", leave=True)
        mem_info = nvmlDeviceGetMemoryInfo(handle)
        print(f"Start of Epoch: {mem_info.total/1e9:.2f} GB, Used: {mem_info.used/1e9:.2f} GB, Free: {mem_info.free/1e9:.2f} GB")
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device).long()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(images)
                log_gpu_stats("Clearing the GPU Memory")
                assert not torch.isnan(outputs).any(), "Model outputs contain NaN!"
                assert not torch.isinf(outputs).any(), "Model outputs contain Inf!"
                loss = criterion(outputs,masks)
                if phase == 'train':
                    optimizer.zero_grad()
                    log_gpu_stats("Clearing the GPU Memory")
                    # print(f"After Loss: {mem_info.total/1e9:.2f} GB, Used: {mem_info.used/1e9:.2f} GB, Free: {mem_info.free/1e9:.2f} GB")
                    loss.backward()
                    log_gpu_stats("Clearing the GPU Memory")
                    optimizer.step()

            # Get predictions and metrics
            preds = torch.argmax(outputs, dim=1)
            batch_dice = dice_score(preds, masks)
            batch_iou = iou_score(preds, masks)

            # Update metrics
            running_loss.append(loss.item())
            running_dice.append(batch_dice)
            running_iou.append(batch_iou)

            # Calculate current averages
            curr_loss = np.mean(running_loss)
            curr_dice = np.mean(running_dice)
            curr_iou = np.mean(running_iou)
            
            log_gpu_stats("Clearing the GPU Memory")
            mem_info = nvmlDeviceGetMemoryInfo(handle)
            # Update progress bar
            pbar.set_postfix({
                'Loss': f'{curr_loss:.4f}',
                'Dice': f'{curr_dice:.4f}',
                'IoU': f'{curr_iou:.4f}',
                'GPU Used': f'{mem_info.used/1e9:.2f} GB',
                'GPU Free': f'{mem_info.free/1e9:.2f} GB'
            })
        log_gpu_stats("Clearing the GPU Memory")
        mem_info = nvmlDeviceGetMemoryInfo(handle)
        print(f"After Training: {mem_info.total/1e9:.2f} GB, Used: {mem_info.used/1e9:.2f} GB, Free: {mem_info.free/1e9:.2f} GB")
        log_gpu_stats("Clearing the GPU Memory")

        # End of batch loop

        # Save epoch metrics
        epoch_loss = np.mean(running_loss)
        epoch_dice = np.mean(running_dice)
        epoch_iou = np.mean(running_iou)
    
        results[f'{phase}_loss'].append(epoch_loss)
        results[f'{phase}_dice'].append(epoch_dice)
        results[f'{phase}_iou'].append(epoch_iou)

        print(f"{phase.upper()} Epoch Avg | Loss: {epoch_loss:.4f} | Dice: {epoch_dice:.4f} | IoU: {epoch_iou:.4f} ")
    
    print()
    
final_path = f"/kaggle/working/best.pth"
torch.save(model.state_dict(), final_path)
print(f"✅ Saved final model: {final_path}")

In [ ]:
import pickle

# Save dictionary
with open('ModelTrainingResults.pkl', 'wb') as f:
    pickle.dump(results, f)


In [ ]:
selected_keys = ['train_loss', 'val_loss', 'train_dice', 'val_dice', 'train_iou', 'val_iou']
import pandas as pd
# Create DataFrame with only the selected keys
df = pd.DataFrame({k: results[k] for k in selected_keys})
df
df.to_csv('ModelTrainingResults.csv', index=False)

# Testing

In [ ]:
# model.load_state_dict(torch.load('/kaggle/input/lung-seg-iit-kgp/pytorch/default/2/best.pth'))

# import os
# import torch
# import numpy as np
# import nibabel as nib
# import cv2
# from tqdm import tqdm
# from torch.nn.functional import softmax

# def preprocess_slice(slice_2d):
#     resized = cv2.resize(slice_2d, (256, 256), interpolation=cv2.INTER_AREA)
#     normalized = resized.astype(np.float32) / 255.0
#     tensor = torch.FloatTensor(normalized).unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
#     return tensor  # [1, 1, 256, 256]

# def predict_3d_mask_from_nii(model, nii_path, device='cuda'):
#     # Load 3D NIfTI scan
#     nii = nib.load(nii_path)
#     image_3d = nii.get_fdata()  # [H, W, D]
#     affine = nii.affine
#     header = nii.header

#     predicted_slices = []

#     model.eval()
#     model.to(device)

#     with torch.no_grad():
#         for i in tqdm(range(image_3d.shape[2]), desc=f"Predicting {os.path.basename(nii_path)}"):
#             slice_2d = image_3d[:, :, i]

#             # If slice is constant/empty, skip prediction (optional)
#             if np.max(slice_2d) == 0:
#                 pred_mask = np.zeros((256, 256), dtype=np.uint8)
#             else:
#                 input_tensor = preprocess_slice(slice_2d).to(device)
#                 output = model(input_tensor)  # [1, 3, 256, 256]
#                 pred_class = torch.argmax(softmax(output, dim=1), dim=1).cpu().numpy()[0]
#                 pred_mask = pred_class.astype(np.uint8)

#             predicted_slices.append(pred_mask)

#     # Stack into 3D volume: [D, H, W] → [H, W, D]
#     predicted_3d = np.stack(predicted_slices, axis=-1)

#     # Save prediction as .nii file
#     pred_nii = nib.Nifti1Image(predicted_3d, affine=affine, header=header)
#     out_path = os.path.join("predicted_nifti", os.path.basename(nii_path).replace('.nii', '_pred.nii'))
#     os.makedirs("predicted_nifti", exist_ok=True)
#     nib.save(pred_nii, out_path)

#     print(f"✅ Saved prediction: {out_path}")
#     return out_path
# out = predict_3d_mask_from_nii(model, '/kaggle/input/medical-segmentation-decathlon-lung/imagesTr/lung_014.nii')

# import nibabel as nib
# import numpy as np
# from skimage import measure
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# def extract_mesh_from_nii(nii_file, threshold=0.5):
#     nii = nib.load(nii_file)
#     volume = nii.get_fdata()
#     binary = (volume > threshold).astype(np.uint8)
#     verts, faces, _, _ = measure.marching_cubes(binary, level=0)
#     return verts, faces

# def plot_two_meshes(gt_nii_path, pred_nii_path):
#     gt_verts, gt_faces = extract_mesh_from_nii(gt_nii_path)
#     pred_verts, pred_faces = extract_mesh_from_nii(pred_nii_path)

#     fig = plt.figure(figsize=(10, 10))
#     ax = fig.add_subplot(111, projection='3d')

#     # Ground Truth Mesh - Blue
#     gt_mesh = Poly3DCollection(gt_verts[gt_faces], alpha=0.4)
#     gt_mesh.set_facecolor('blue')
#     ax.add_collection3d(gt_mesh)

#     # Predicted Mask Mesh - Red
#     pred_mesh = Poly3DCollection(pred_verts[pred_faces], alpha=0.4)
#     pred_mesh.set_facecolor('red')
#     ax.add_collection3d(pred_mesh)

#     # Set axis limits based on larger volume
#     all_verts = np.vstack((gt_verts, pred_verts))
#     ax.set_xlim(0, np.max(all_verts[:, 0]))
#     ax.set_ylim(0, np.max(all_verts[:, 1]))
#     ax.set_zlim(0, np.max(all_verts[:, 2]))

#     ax.set_title("Ground Truth (Blue) vs Predicted Mask (Red)")
#     plt.tight_layout()
#     plt.show()

# # Example usage
# plot_two_meshes('/kaggle/input/medical-segmentation-decathlon-lung/labelsTr/lung_014.nii', "/kaggle/working/predicted_nifti/lung_014_pred.nii")
